<a href="https://colab.research.google.com/github/martiitesti-rgb/ShopJournal/blob/main/extracting_cues.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
import pandas as pd

nltk.download('vader_lexicon')

analyzer = SentimentIntensityAnalyzer()
def extract_cues(query, notes):

    combined_text = f"{query} {notes}".strip().lower()
    if not combined_text:
        return 0.0, "neutral", {"urgency": False, "budget": False, "diet": False, "gift": False}, []

    scores = analyzer.polarity_scores(combined_text)
    compound_score = scores['compound']

    if compound_score >= 0.05:
        sentiment_label = "positive"
    elif compound_score <= -0.05:
        sentiment_label = "negative"
    else:
        sentiment_label = "neutral"

    intent_keywords = {
        "urgency": ["urgent", "fast", "quick", "now", "today", "tomorrow", "last minute","emergency"],
        "budget": ["cheap", "budget","not expensive", "affordable", "discount", "under", "max", "sale", "price"],
        "diet": ["vegetarian", "vegan",  "gluten-free", "allergy", "healthy", "diet"],
        "gift": ["gift", "present", "birthday", "for my", "anniversary", "mom", "dad", "friend"]
    }

    keywords_found=[]
    intent_flags = {}
    for intent, keywords in intent_keywords.items():
      trovato = False
      for keyword in keywords:
        if keyword in combined_text:
            keywords_found.append(keyword)
            trovato = True
            break
      intent_flags[intent] = trovato
    return compound_score, sentiment_label, intent_flags, keywords_found



[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


In [3]:
#old version
print("\nTEST 1\n");
test1_query="i need a pair of shoes"
test1_note="i need them today since it's a birthday present, it's better if they are not expensive"

compound_score, sentiment_label, intent_flags, keywords_found = extract_cues(test1_query, test1_note)
print(f"\nCompound score '{compound_score}', sentiment '{sentiment_label}', flags: '{intent_flags}', keywords: '{ keywords_found}'");

print("\nTEST 2\n");
test2_query="i need some gorgonzola"
test2_note="it's for a vegetarian recipe, i hope it's on sale"

compound_score, sentiment_label, intent_flags, keywords_found = extract_cues(test2_query, test2_note)
print(f"\nCompound score '{compound_score}', sentiment '{sentiment_label}', flags: '{intent_flags}', keywords: '{ keywords_found}'");

print("\nTEST 3\n");
test3_query="i want to buy a present for my mother"
test3_note="it's her birthday next week and she hates not receiving anything"

compound_score, sentiment_label, intent_flags, keywords_found = extract_cues(test3_query, test3_note)
print(f"\nCompound score '{compound_score}', sentiment '{sentiment_label}', flags: '{intent_flags}', keywords: '{ keywords_found}'");

print("\nTEST 4\n");
test4_query="I am so excited to buy a new dress for my birthday"
test4_note="I want something affordable but elegant and i need it now"

compound_score, sentiment_label, intent_flags, keywords_found = extract_cues(test4_query, test4_note)
print(f"\nCompound score '{compound_score}', sentiment '{sentiment_label}', flags: '{intent_flags}', keywords: '{ keywords_found}'");

print("\nTEST 5\n");
test5_query="i want a new pair of affordable headphones "
test5_note="I hate the ones that break easily, last time they were terrible "

compound_score, sentiment_label, intent_flags, keywords_found = extract_cues(test5_query, test5_note)
print(f"\nCompound score '{compound_score}', sentiment '{sentiment_label}', flags: '{intent_flags}', keywords: '{ keywords_found}'\n");




TEST 1


Compound score '0.4404', sentiment 'positive', flags: '{'urgency': True, 'budget': True, 'diet': False, 'gift': True}', keywords: '['today', 'not expensive', 'present']'

TEST 2


Compound score '0.4404', sentiment 'positive', flags: '{'urgency': False, 'budget': True, 'diet': True, 'gift': False}', keywords: '['sale', 'vegetarian']'

TEST 3


Compound score '-0.3818', sentiment 'negative', flags: '{'urgency': False, 'budget': False, 'diet': False, 'gift': True}', keywords: '['present']'

TEST 4


Compound score '0.7308', sentiment 'positive', flags: '{'urgency': True, 'budget': True, 'diet': False, 'gift': True}', keywords: '['now', 'affordable', 'birthday']'

TEST 5


Compound score '-0.6249', sentiment 'negative', flags: '{'urgency': False, 'budget': True, 'diet': False, 'gift': False}', keywords: '['affordable']'



In [11]:
test_suite = [
    {
        "name": "Test 1",
        "query": "i need a pair of shoes",
        "notes": "i need them today since it's a birthday present, it's better if they are not expensive",
        "expected_flags": {"urgency": True, "budget": True, "diet": False, "gift": True},
        "expected_sentiment": "positive"
    },
    {
        "name": "Test 2: Dieta e Budget",
        "query": "i need some gorgonzola",
        "notes": "it's for a vegetarian recipe, i hope it's on sale",
        "expected_flags": {"urgency": False, "budget": True, "diet": True, "gift": False},
        "expected_sentiment": "positive"
    },
    {
        "name": "Test 3: Regalo con sentiment negativo",
        "query": "i want to buy a present for my mother",
        "notes": "it's her birthday next week and she hates not receiving anything",
        "expected_flags": {"urgency": False, "budget": False, "diet": False, "gift": True},
        "expected_sentiment": "negative"
    },
    {
        # Ambiguity: it proves that the keyword approach is limited due to the fact that altough
        #there are some keyword present in the dictionary, the aim of this phrase is not to buy a birthday present
        "name": "Test 4",
        "query": "I am so excited to buy a new dress for my birthday",
        "notes": "I want something affordable but elegant and i need it now",
        "expected_flags": {"urgency": True, "budget": True, "diet": False, "gift": True},
        "expected_sentiment": "positive"
    },
    {
        "name": "Test 5",
        "query": "i want a new pair of affordable headphones",
        "notes": "I hate the ones that break easily, last time they were terrible",
        "expected_flags": {"urgency": False, "budget": True, "diet": False, "gift": False},
        "expected_sentiment": "negative"
    },
    {
        #input vuoto
        "name": "Test 6:",
        "query": "",
        "notes": "",
        "expected_flags": {"urgency": False, "budget": False, "diet": False, "gift": False},
        "expected_sentiment": "neutral"
    },
    {
        #test without keywords
        "name": "Test 7:",
        "query": "i need a cellphone",
        "notes": "the old one is broken",
        "expected_flags": {"urgency": False, "budget": False, "diet": False, "gift": False},
        "expected_sentiment": "negative" #?
    },
     {
        #test ambiguo
        "name": "Test 8:",
        "query": "i need a cellphone",
        "notes": "the old phone was a cheap gift, so the next one must not be cheap",
        #"expected_flags": {"urgency": False, "budget": False, "diet": False, "gift": False}, # this should be the correct ones but it will probably be a
        #false positive returning
        "expected_flags":{"urgency": False, "budget": True, "diet": False, "gift": True},
        "expected_sentiment": "positive" #?
    }
]

for test in test_suite:
    compound, label, flags, keywords = extract_cues(test["query"], test["notes"])

    assert flags == test["expected_flags"], f"{test['name']} has wrong flags . \n right: {test['expected_flags']}\n wrong: {flags}"
    assert label == test["expected_sentiment"], f"{test['name']} has  a wrong sentiment. right: {test['expected_sentiment']}, wrong: {label}"

    print(f"{test['name']} ok: keywords : {keywords})")



Test 1 ok: keywords : ['today', 'not expensive', 'present'])
Test 2: Dieta e Budget ok: keywords : ['sale', 'vegetarian'])
Test 3: Regalo con sentiment negativo ok: keywords : ['present'])
Test 4 ok: keywords : ['now', 'affordable', 'birthday'])
Test 5 ok: keywords : ['affordable'])
Test 6: ok: keywords : [])
Test 7: ok: keywords : [])
Test 8: ok: keywords : ['cheap', 'gift'])
